# Stage 4: checking the partnership data

**What this stage does:** makes sure the partnership table is right before we build any network.

We run the structural checks and the scorecard checks from `library/lib_03_validation.ipynb` and write all
results to `docs/data_quality_report.md`.

**Stop rule:** if any check fails, or if the extractor raised a warning, this notebook stops with an error, so
the later stages never run on wrong data.

**Reads:** the tables from Stages 2 and 3, and the match files
**Writes:** `docs/data_quality_report.md`, `outputs/tables/validation_*.csv` and `outputs/tables/dataset_counts.csv`

In [1]:
%run ./library/lib_00_settings.ipynb
%run ./library/lib_01_data_loading.ipynb
%run ./library/lib_03_validation.ipynb
stage_log = get_stage_logger("04_validate")

In [2]:
included_matches = read_table(PROCESSED_FOLDER / "match_metadata.csv")
observations = read_table(PROCESSED_FOLDER / "partnership_observations.csv")
innings_table = read_table(PROCESSED_FOLDER / "innings_summary.csv")
extraction_warnings = read_table(PROCESSED_FOLDER / "extraction_warnings.csv")
file_audit = read_table(INTERIM_FOLDER / "file_audit.csv")

### 1. Run the checks

In [3]:
structural_results = pd.DataFrame(observation_table_checks(observations, innings_table),
                                  columns=["check", "passed", "detail"])
scorecard_results = pd.DataFrame([
    check_row
    for match_record in included_matches.to_dict("records")
    for check_row in scorecard_checks(load_match_file(RAW_MATCH_FOLDER / match_record["filename"]),
                                      match_record["match_id"], match_record["format"])])

write_table(structural_results, TABLES_FOLDER / "validation_structural_checks.csv")
write_table(scorecard_results, TABLES_FOLDER / "validation_scorecard_checks.csv")
structural_results

,check,passed,detail
0,innings_partnership_runs_equal_delivery_runs,True,"350 innings, 84983 runs"
1,distinct_players_in_pair,True,
2,canonical_pair_order,True,
3,all_player_ids_resolved,True,
4,runs_non_negative_integers,True,
5,runs_equal_batter_plus_extras,True,
6,every_observation_has_a_delivery,True,
7,sequence_numbers_contiguous,True,
8,innings_end_only_on_last_observation,True,
9,observation_count_matches_innings_table,True,2828 partnerships


### 2. Counts per format

In [4]:
qualifying_innings = innings_table[innings_table["qualifying"]]
dataset_counts = pd.DataFrame({
    "matches": included_matches.groupby("format").size(),
    "matches_with_england_innings": qualifying_innings.groupby("format")["match_id"].nunique(),
    "england_innings": qualifying_innings.groupby("format").size(),
    "observations": observations.groupby("format").size(),
    "partnership_runs": observations.groupby("format")["partnership_runs"].sum(),
    "wicket_records": qualifying_innings.groupby("format")["wicket_records"].sum(),
    "deliveries": qualifying_innings.groupby("format")["delivery_count"].sum(),
}).reindex(list(FORMATS)).reset_index()
write_table(dataset_counts, TABLES_FOLDER / "dataset_counts.csv")
dataset_counts

,format,matches,matches_with_england_innings,england_innings,observations,partnership_runs,wicket_records,deliveries
0,Test,81,81,151,1357,43080,1328,66829
1,ODI,88,85,85,708,22929,662,22766
2,T20I,118,114,114,763,18974,675,12878


### 3. Write the data-quality report

The report is rebuilt from scratch on every run, so it always matches the current data.

In [5]:
scorecard_summary = (scorecard_results.groupby(["format", "check"])["passed"]
                     .agg(checked="size", passed="sum").reset_index())
failed_scorecard_checks = scorecard_results[~scorecard_results["passed"]]
special_innings = innings_table[(innings_table["penalty_runs_pre"] + innings_table["penalty_runs_post"] > 0)
                                | innings_table["absent_hurt"].notna() | innings_table["super_over"]]
other_endings = observations[observations["boundary_reason"].str.startswith(("replacement", "retirement", "unexplained"))]
matches_without_england_innings = included_matches[included_matches["n_innings_england"] == 0]
unreadable_files = int(file_audit["exclusion_reason"].astype(str).str.startswith("unreadable").sum())

report_lines = [
    "# Data-quality report", "",
    "This file is written by `notebooks/04_validate_observations.ipynb` every time the pipeline runs.", "",
    "## Source files", "",
    f"- Match files in the archive: {len(file_audit)} (unreadable: {unreadable_files})",
    f"- Matches kept: {int(file_audit['included'].sum())}; left out: {int((~file_audit['included']).sum())}",
    "", "Why files were left out (one reason per file):", "",
    markdown_table(file_audit.loc[~file_audit["included"], "exclusion_reason"].value_counts()
                   .rename_axis("reason").reset_index(name="files")),
    "", "## Counts per format", "", markdown_table(dataset_counts), "",
    f"Matches kept in which England did not bat (washed out or finished before England's innings): "
    f"{len(matches_without_england_innings)} ({', '.join(matches_without_england_innings['match_id'].astype(str))}). "
    "They have no England innings, so they add no partnerships.",
    "", "## Structural checks", "", markdown_table(structural_results), "",
    "## Scorecard checks", "",
    "Each innings total is added up straight from the balls in the match file, without the partnership code, "
    "and compared with the target and the winning margin in runs that Cricsheet stores in the same file. "
    "Matches decided by a rain rule are not part of this check, because their target comes from the rain rule.", "",
    markdown_table(scorecard_summary), "",
    f"Scorecard checks that failed: {len(failed_scorecard_checks)}", "",
    markdown_table(failed_scorecard_checks) if len(failed_scorecard_checks) else "", "",
    "## Extraction warnings", "",
    markdown_table(extraction_warnings) if len(extraction_warnings) else
    "None. Every change of pair is explained by a wicket, a recorded batter replacement or the end of the innings.", "",
    "## Partnerships that ended in another way", "",
    "Partnerships that ended with a retirement or a recorded replacement instead of a dismissal.", "",
    markdown_table(other_endings[["match_id", "format", "match_date", "innings_index", "player_1_name",
                                  "player_2_name", "partnership_runs", "boundary_reason"]]), "",
    "## Innings with penalty runs, absent batters or super overs", "",
    markdown_table(special_innings[["match_id", "format", "innings_index", "penalty_runs_pre", "penalty_runs_post",
                                    "absent_hurt", "super_over", "runs_from_deliveries"]].fillna("")), "",
    "Penalty runs given to a whole innings are not part of any ball, so they are not given to any pair.", "",
]
(PROJECT_ROOT / "docs" / "data_quality_report.md").write_text("\n".join(report_lines), encoding="utf-8")

3345

### 4. Stop rule

In [6]:
all_checks_passed = bool(structural_results["passed"].all() and scorecard_results["passed"].all()
                         and len(extraction_warnings) == 0)
stage_log.info("structural checks passed: %d/%d", structural_results["passed"].sum(), len(structural_results))
stage_log.info("scorecard checks passed: %d/%d", scorecard_results["passed"].sum(), len(scorecard_results))
stage_log.info("extraction warnings: %d", len(extraction_warnings))
stage_log.info("CHECKS %s", "PASSED" if all_checks_passed else "FAILED, see docs/data_quality_report.md")
assert all_checks_passed, "Some data checks failed, see docs/data_quality_report.md"

2026-11-19 18:05:07,911 INFO partnerships.04_validate: structural checks passed: 13/13


2026-11-19 18:05:07,911 INFO partnerships.04_validate: scorecard checks passed: 318/318


2026-11-19 18:05:07,911 INFO partnerships.04_validate: extraction warnings: 0


2026-11-19 18:05:07,912 INFO partnerships.04_validate: CHECKS PASSED
